# GB.Cell Quickstart

This demo walks through installing GB.Cell and embedding new single cell data.

__Requirements__:
- GPU recommended (A100 or equivalent) but CPU is supported

### Install the standalone GB.Cell package

In [ ]:
!pip install -e ".[flash_attn]"

### Grab some data from GEO and load into anndata

In [ ]:
%%bash
mkdir -p data
cd data
wget -nv -O GSE214695.tar 'http://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE214695&format=file'
tar -xvf GSE214695.tar
cd ..

In [ ]:
import scanpy as sc

adata = sc.read_10x_mtx('data', prefix='GSM6614348_HC-1_')
sc.pp.filter_cells(adata, min_genes=500)
sc.pp.filter_genes(adata, min_cells=3)
# No more normalization needed, GB.Cell uses raw counts

### Preprocess the anndata for GB.Cell

In [ ]:
from gb_cell.utils import align_adata, preprocess_counts

aligned_adata, attention_mask = align_adata(adata)

### Get GB.Cell embeddings

In [ ]:
import numpy as np
import torch
from gb_cell.models import CellFoundationModel, CellFoundationConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 2

config = CellFoundationConfig.from_pretrained('genbio-ai/GB.Cell-3M')
model = CellFoundationModel.from_pretrained('genbio-ai/GB.Cell-3M', config=config)
model = model.to(device).eval()
if device == 'cuda':
    model = model.to(torch.bfloat16)

batch_counts = aligned_adata[:batch_size].X.toarray()
batch_input = preprocess_counts(batch_counts, device=device)

# Build attention mask with 2 extra positions for depth tokens
attn_mask = torch.from_numpy(attention_mask).unsqueeze(0).repeat(batch_size, 1).to(device)
depth_mask = torch.ones((batch_size, 2), device=device)
attn_mask = torch.cat([attn_mask, depth_mask], dim=1)

with torch.no_grad():
    outputs = model(
        input_ids=batch_input,
        attention_mask=attn_mask,
        output_hidden_states=True,
    )

# Per-gene embeddings
last_hidden = outputs.last_hidden_state[:, :-2, :]  # drop depth tokens
print('Full embedding shape (batch, genes, hidden):', last_hidden.shape)

# Non-zero genes only
nz_embs = last_hidden[:, attention_mask.astype(bool), :]
print('Non-zero genes embedding shape:', nz_embs.shape)

# Mean-pool for a single cell-level vector
mask_exp = attn_mask[:, :-2].unsqueeze(-1).float()
cell_embs = (last_hidden * mask_exp).sum(1) / mask_exp.sum(1)
print('Cell-level embedding shape:', cell_embs.shape)